# Development workflows on Databricks

**Purpose:** There are multiple ways of writing and running code on Databricks, each with their own advantages. This notebook describes these approaches and gives a rule for when to use each. 

**What this compares.**
1. Interactive workspace: notebooks attached to a cluster, in the browser.
2. Local IDE + Databricks Connect: VS Code and your editor's tooling, remote Spark.
3. Asset Bundle deploy: versioned jobs and pipelines shipped via `databricks bundle`. This will also be covered in more detail in `05_asset_bundles`

The point of all three is that one source file can travel the whole way:

1. Sketch it in a workspace notebook.
2. Pull it local with the VS Code extension, refactor logic into `src/`, add `pytest`, and run against remote Spark via Connect.
3. Reference it from a job in `databricks.yml` and deploy the bundle. The same file now runs as a governed, scheduled production job.

| Dimension | Interactive workspace | VS Code + Connect | Asset Bundle deploy |
|---|---|---|---|
| Speed to first result | **Fastest** | Medium | Slowest to set up |
| Reproducibility | Low | Medium | **High** |
| Governance / audit | Low | Medium | **High** |
| Git + code review | Awkward | **Native** | **Native** |
| Unit testing | Hard | **Easy** | **Easy** |
| Best for | Exploration, demos | Engineering, tests | Production, CI/CD |


## Compute strategy: match the cluster to the workload

| Compute | Reach for it when | Cost shape | The catch |
|---|---|---|---|
| Serverless | Ad-hoc queries, prototyping, instant-on | Pay per active execution | No init scripts or custom cluster config |
| All-purpose | Connected IDE (Connect) or live notebook authoring | Accrues continuously while running | Priciest per DBU; easy to leave running |
| Job compute | Scheduled, long, or reproducible runs | Cheapest DBUs, ephemeral | Cold start spins up per run |

Default to the **Databricks Runtime for ML (DBR ML)**: it ships tuned MLflow, PyTorch, and distributed-training libraries, so you skip most `pip install` friction. Use a **single node** for `pandas`/`scikit-learn`; reach for **multi-node** only when the work is genuinely PySpark or distributed training. Need cluster scale without rewriting pure Python? Bridge with the [Pandas API on Spark](https://docs.databricks.com/aws/en/pandas/pandas-on-spark).


## (a) Interactive workspace: explore fast

**Use it when** you're exploring a dataset, prototyping a model, debugging interactively, or giving a demo. The feedback loop is the shortest you'll get.

**Notes:**
- **Ephemeral state:** The REPL environment is incredibly fast for iteration, but hidden cell state is the enemy of reproducibility. Always restart the kernel and "run all" before trusting your results.
- **The technical debt trap:** Notebooks are notoriously difficult to properly modularize, lint, or unit-test. The moment core logic proves valuable, refactor it out of the notebook and into an importable Python package.
- **Version control limits:** Databricks Git Folders are sufficient for basic syncing, but the browser UI lacks the robust tooling needed to handle complex merge conflicts, rebasing, or multi-branch tracking. On collaborative enterprise projects, relying exclusively on the workspace UI creates severe friction; transitioning to a local IDE with native `.git` management becomes mandatory.

In [0]:
# --- 1. STATE MANAGEMENT ---
# The workspace REPL hides state. If things act weird, uncomment and run this to nuke the Python process.
# dbutils.library.restartPython()

# If you are actively developing local packages (e.g., in our src/ directory) via workspace files,
# autoreload forces the notebook to pick up your module changes without restarting the cluster.
%load_ext autoreload
%autoreload 2

# --- 2. PARAMETERIZATION ---
# Never hardcode environment strings or limits. 
# dbutils.widgets creates UI inputs at the top of the notebook so anyone can toggle them safely.
dbutils.widgets.dropdown("env", "dev", ["dev", "staging", "prod"], "Environment")
dbutils.widgets.text("sample_size", "1000", "Churn Sample Size")

current_env = dbutils.widgets.get("env")
n_samples = int(dbutils.widgets.get("sample_size"))

print(f"Target Environment: {current_env.upper()} | Generating {n_samples} synthetic churn records...")


In [0]:
# --- 3. DATA LOADING (Customer Churn Spine) ---
from src.utils.data import upload_to_uc_volume

# 1. Load locally
pdf = pd.read_csv("data/telco_customer_churn.csv")
#df_churn = spark.read.csv("data/telco_customer_churn.csv")

# 2. Convert from Pandas to SparakPush to remote Spark cluster
df_churn = spark.createDataFrame(pdf)

# 3. Save as a governed Delta table (No Volume required)
target_table = "main.default.telco_customer_churn" # Update catalog/schema
df_churn.write.format("delta").mode("overwrite").saveAsTable(target_table)

#df_churn = spark.read.table('main.default.telco_customer_churn')

display(df_churn)



**PySpark execution model:** Transformations like `.withColumn()`, `.filter()`, and `.select()` build a logical query plan without executing anything. Actions like `.count()`, `.show()`, `.collect()`, and `.toPandas()` trigger the execution engine to run the plan and return results.

In [0]:
row_count = df_churn.count()  # Triggers full execution
print(f"Churn DataFrame rows: {row_count}")

# Show basic head
print("First 5 rows:")
df_churn_mock.show(5)

# Cache for repeated access (be cautious; only for reused results)
df_churn_mock.cache() # not supported on serverless

In [0]:
# PySpark vs pandas: conversion
# Only do this for small, filtered DataFrames
df_churn_sample = df_churn_mock.limit(50)  # Safe small sample
pandas_df = df_churn_sample.toPandas()
print("Converted to pandas:")
print(pandas_df.head())

# Back to PySpark
df_churn_from_pd = spark.createDataFrame(pandas_df)
print(f"Converted back to Spark DataFrame; rows: {df_churn_from_pd.count()}")



## (b) Local IDE + Databricks Connect

**Use it when** the code matters. If a script needs unit tests, rigorous peer review, or a production schedule, it belongs here.

**The Engineering Standard:**
- **Native Git:** Escapes the limits of Workspace Git Folders. A real terminal handles multi-branch workflows, rebasing, and complex conflict resolution.
- **Refactoring Ergonomics:** You *can* author `.py` files in the browser via Workspace Files, but building out a real `src/` package is awkward there. A local IDE adds jump-to-definition and makes cross-file refactoring far easier.
- **CI/CD Tooling:** Unlocks the standard local stack: `pytest`, linters (`ruff`, `flake8`), type-checking (`mypy`), and pre-commit hooks.

**Authentication:** Never hardcode tokens. Prefer OAuth user-to-machine over legacy Personal Access Tokens (PATs). Run this locally to configure your `~/.databrickscfg` profile, which both the IDE and the CLI read:

```bash
databricks auth login --host https://<your-workspace-host>
# creates or updates a named profile; the VS Code extension and CLI both read it


In [0]:
from src.local_utils import get_spark, get_parameter

# Usage: This single line now works in the browser, in VS Code, and in a CI/CD pipeline.
my_spark = get_spark()
print(f"Active Spark Version: {my_spark.version}")

# Safely fetch the target environment without breaking local tests
target_catalog = get_parameter("catalog_name", default="dev_catalog")

In [0]:
# --- DATABRICKS CONNECT SANITY CHECK ---
# Guarded so it never hard-fails. 
# In a workspace notebook, you already have a `spark` session and don't need this.
# From a local IDE, Connect builds a session that runs on your remote Serverless/Cluster.

try:
    from databricks.connect import DatabricksSession

    spark = DatabricksSession.builder.getOrCreate()
    # Using range().count() forces actual remote compute execution, proving the link is live.
    print("Remote Spark reachable:", spark.range(3).count(), "rows computed remotely.")
except Exception as exc:  # ImportError locally, or auth/config errors
    print(f"Databricks Connect not active here ({type(exc).__name__}).")
    print("Expected on GitHub or without a configured profile.")
    print("Set up: databricks auth login, then point the profile at a cluster or serverless.")

In [0]:
# --- SPARK READ VIA UC VOLUME ---
# If the dataset is large, pushing it through local memory will crash.
# The standard is to upload to a cloud volume first, then let Spark read natively.
from src.utils.data import upload_to_uc_volume

volume_path = upload_to_uc_volume(
    catalog="main", 
    schema="default", 
    volume="raw", 
    local_file_path="data/telco_customer_churn.csv"
)

# ingestion: remote spark reads the raw CSV from the volume
df_spark = spark.read.csv(volume_path, header=True, inferSchema=True)

# UC Table option: 
# target_table = "main.default.telco_customer_churn"
# df_spark_native.write.format("delta").mode("overwrite").saveAsTable(target_table)
# df_spark = spark.read.table(target_table)
# print(f"Success: Retrieved {df_spark.count()} records remotely from Unity Catalog.")

display(df_spark.limit(5))

## (c) Asset Bundle deploy: ship to production

A [Databricks Asset Bundle (DAB)](https://docs.databricks.com/dev-tools/bundles/index.html) packages your notebooks, `src/` modules, and ML infrastructure as versioned, declarative configuration (`databricks.yml`). It ensures your pipeline deploys identically across `dev`, `staging`, and `prod` workspaces.

**Use it when** the work leaves your laptop. If a pipeline needs to run on a schedule, trigger via an event, or pass through CI/CD, it must be bundled. **No more UI click-ops.**

**The Engineering Standard:**
- **Infrastructure as Code (IaC):** Workspace state (Jobs, DLT Pipelines, ML Endpoints) becomes a strict function of the `main` branch in Git.
- **Environment Promotion:** Code is tested locally, deployed to a `staging` workspace via GitHub Actions, and finally promoted to `prod` upon approval.


```bash
databricks bundle validate -t dev      # check config against the schema
databricks bundle deploy -t staging  # create/update jobs + pipelines in staging
databricks bundle run churn_training -t staging
```

*(Shown for shape, not executed here.)*

In [0]:
# When deployed via Asset Bundles, Jobs inject parameters into the environment

def get_job_parameter(key: str, default: str = "local_dev") -> str:
    """Retrieves parameters injected by the databricks.yml bundle configuration."""
    try:
        # In a deployed Bundle/Job, dbutils is globally available
        from dbruntime.dbutils import DBUtils
        dbutils = DBUtils(spark)
        return dbutils.widgets.get(key)
    except (ImportError, NameError, Exception):
        # Graceful fallback for local IDE execution
        return default

# The 'env' parameter maps to ${bundle.environment} in the YAML definition
current_env = get_job_parameter("env", default="dev")

print(f"Pipeline executing in environment: {current_env.upper()}")
if current_env == "prod":
    print("WARNING: Production environment detected. Writing to prod catalogs.")

## Further reading

[`docs/links.md`](docs/links.md) carries the full annotated list, one line of *why* per entry. The essentials for this notebook:

- [Databricks extension for VS Code](https://docs.databricks.com/dev-tools/vscode-ext/index.html)
- [Databricks Connect](https://docs.databricks.com/dev-tools/databricks-connect/index.html)
- [Authentication (OAuth / profiles)](https://docs.databricks.com/dev-tools/auth/index.html)
- [Asset Bundles](https://docs.databricks.com/dev-tools/bundles/index.html)